# Inference Playground

Plug-and-play inference on any audio folder. Automatically:
1. Scans folder for audio files
2. Loads or creates transcriptions, text features, WavLM features, wav2vec2 scores
3. Runs weighted ensemble prediction
4. If GT labels exist, computes metrics

**Set weights to 0 to skip that model entirely** (won't compute features if not needed).

In [ ]:
# ================================================================
# CONFIGURATION -- edit these paths and weights
# ================================================================

# Input folder containing audio files
AUDIO_FOLDER = r"audios2"  # e.g. "audios2", "audios4", or full path

# -- Stacked ensemble model paths -----------------------------
WAVLM_XGBOOST_MODEL = r"checkpoints_trained/xgboost_wavlm.json"
WAVLM_SCALER        = r"checkpoints_trained/scaler_wavlm.pkl"
HYBRID_XGBOOST_MODEL = r"checkpoints_trained/xgboost_hybrid.json"
HYBRID_SCALER        = r"checkpoints_trained/scaler_hybrid.pkl"
META_MODEL           = r"checkpoints_trained/meta_logistic.pkl"

# -- Baseline text model (for comparison, set to None to skip) --
TEXT_XGBOOST_MODEL  = r"checkpoints_trained/xgboost_text.json"
TEXT_SCALER         = r"checkpoints_trained/scaler_text.pkl"

# -- Threshold -------------------------------------------------
THRESHOLD = 0.45  # applied to meta-learner output

# Whisper model for transcription (only used if transcripts don't exist)
WHISPER_MODEL = "small"  # "tiny", "base", "small", "medium"

AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".wma", ".aac", ".webm", ".mp4"}

In [ ]:
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import joblib

warnings.filterwarnings('ignore')

# Validate weights
assert abs(W_TEXT + W_WAVLM + W_WAV2VEC2 - 1.0) < 0.01, \
    f"Weights must sum to 1.0, got {W_TEXT + W_WAVLM + W_WAV2VEC2}"

AUDIO_EXTS = {".wav", ".mp3", ".m4a", ".flac", ".ogg", ".wma", ".aac", ".webm", ".mp4"}

## 1. Scan Folder

In [ ]:
audio_dir = Path(AUDIO_FOLDER).resolve()
folder_name = audio_dir.name
NB_DIR = Path(".").resolve()

# Find all audio files
audio_files = sorted([
    f for f in audio_dir.rglob("*")
    if f.suffix.lower() in AUDIO_EXTS and f.is_file()
])

print(f"Folder: {audio_dir}")
print(f"Notebook dir: {NB_DIR}")
print(f"Audio files found: {len(audio_files)}")
for ext in sorted(set(f.suffix.lower() for f in audio_files)):
    n = sum(1 for f in audio_files if f.suffix.lower() == ext)
    print(f"  {ext}: {n}")

# File list
print(f"\n{'='*60}")
print(f"FILE LIST ({len(audio_files)} files):")
print(f"{'='*60}")
print("filename")
for f in audio_files:
    print(f.name)
print(f"{'='*60}\n")

# Check for GT labels
gt_candidates = [
    NB_DIR / f"{folder_name}GT.csv",
    NB_DIR / "GT.csv",
    NB_DIR / "labels.csv",
    audio_dir / f"{folder_name}GT.csv",
    audio_dir / "GT.csv",
    audio_dir / "labels.csv",
]
GT_CSV = None
for gp in gt_candidates:
    if gp.exists():
        GT_CSV = gp
        break

HAS_GT = GT_CSV is not None
if HAS_GT:
    gt_df = pd.read_csv(GT_CSV)
    print(f"Ground truth: {GT_CSV}")
    print(f"  Columns: {list(gt_df.columns)}")
    print(f"  Rows: {len(gt_df)}")
    label_col = None
    for candidate in ["label_int", "label", "gt", "GT", "ground_truth", "cheating"]:
        if candidate in gt_df.columns:
            label_col = candidate
            break
    if label_col:
        print(f"  Label column: {label_col}")
        print(f"  {gt_df[label_col].value_counts().to_dict()}")
    else:
        print(f"  WARNING: No label column found. Will treat as unlabeled.")
        HAS_GT = False
else:
    print(f"No GT file found (looked for {folder_name}GT.csv). Running without labels.")

# Derived paths
TRANSCRIPTS_JSON = NB_DIR / f"{folder_name}_transcripts.json"
TRANSCRIPTS_CSV  = NB_DIR / f"{folder_name}_transcripts.csv"
FEATURES_CSV     = NB_DIR / f"{folder_name}_features.csv"
WAVLM_FEAT_CSV  = NB_DIR / f"{folder_name}_wavlm.csv"
SBERT_FEAT_CSV   = NB_DIR / f"{folder_name}_features_sbert.csv"

print(f"\nDerived paths (in notebook dir):")
print(f"  Transcripts JSON: {TRANSCRIPTS_JSON} {'[EXISTS]' if TRANSCRIPTS_JSON.exists() else '[MISSING]'}")
print(f"  Text features:    {FEATURES_CSV} {'[EXISTS]' if FEATURES_CSV.exists() else '[MISSING]'}")
print(f"  WavLM features:   {WAVLM_FEAT_CSV} {'[EXISTS]' if WAVLM_FEAT_CSV.exists() else '[MISSING]'}")
print(f"  SBERT features:   {SBERT_FEAT_CSV} {'[EXISTS]' if SBERT_FEAT_CSV.exists() else '[MISSING]'}")

## 2. Transcription (if needed)
Skipped if transcripts JSON already exists.

In [ ]:
need_transcripts = not TRANSCRIPTS_JSON.exists()

# Prompt that nudges Whisper to preserve filler words instead of stripping them
FILLER_PROMPT = "um, uh, hmm, uh-huh, like, you know, I mean, so, actually, basically"

if need_transcripts:
    from faster_whisper import WhisperModel
    import librosa

    print(f"Transcribing {len(audio_files)} files with faster-whisper ({WHISPER_MODEL}, int8)...")
    print(f"Using filler-preserving prompt: '{FILLER_PROMPT[:60]}...'")
    model_w = WhisperModel(WHISPER_MODEL, device="cpu", compute_type="int8")

    transcripts = {}
    for fp in tqdm(audio_files, desc="Transcribing"):
        try:
            segments, info = model_w.transcribe(
                str(fp), language="en", word_timestamps=True,
                initial_prompt=FILLER_PROMPT,
            )
            words = []
            text_parts = []
            for seg in segments:
                text_parts.append(seg.text.strip())
                for w in (seg.words or []):
                    words.append({
                        "word": w.word.strip(),
                        "start": round(w.start, 3),
                        "end": round(w.end, 3),
                    })
            duration = librosa.get_duration(path=str(fp))
            transcripts[fp.name] = {
                "filename": fp.name,
                "text": " ".join(text_parts),
                "words": words,
                "duration_sec": round(duration, 2),
            }
        except Exception as e:
            print(f"  Failed: {fp.name}: {e}")
            transcripts[fp.name] = {
                "filename": fp.name, "text": "", "words": [], "duration_sec": 0,
            }

    with open(TRANSCRIPTS_JSON, "w", encoding="utf-8") as f:
        json.dump(transcripts, f, indent=2, ensure_ascii=False)
    print(f"Saved JSON: {TRANSCRIPTS_JSON}")

    csv_rows = []
    for fn, t in transcripts.items():
        csv_rows.append({
            "filename": t["filename"],
            "text": t["text"],
            "n_words": len(t["words"]),
            "duration_sec": t["duration_sec"],
        })
    pd.DataFrame(csv_rows).to_csv(TRANSCRIPTS_CSV, index=False)
    print(f"Saved CSV:  {TRANSCRIPTS_CSV}")

    del model_w
else:
    print(f"Transcripts already exist: {TRANSCRIPTS_JSON}")
    if not TRANSCRIPTS_CSV.exists():
        with open(TRANSCRIPTS_JSON, encoding="utf-8") as f:
            transcripts = json.load(f)
        csv_rows = []
        for fn, t in transcripts.items():
            csv_rows.append({
                "filename": t.get("filename", fn),
                "text": t.get("text", ""),
                "n_words": len(t.get("words", [])),
                "duration_sec": t.get("duration_sec", 0),
            })
        pd.DataFrame(csv_rows).to_csv(TRANSCRIPTS_CSV, index=False)
        print(f"Generated CSV from existing JSON: {TRANSCRIPTS_CSV}")

## 3. Text + Pause + Prosodic Features (if needed)
Skipped if features CSV already exists. Needed for the hybrid model.

In [ ]:
need_text_features = not FEATURES_CSV.exists()

if need_text_features:
    nb_dir_str = str(NB_DIR)
    if nb_dir_str not in sys.path:
        sys.path.insert(0, nb_dir_str)
    from extract_features_company import (
        compute_text_features, compute_pause_features, compute_prosodic_features,
        _empty_text_features, _empty_pause_features, _empty_prosodic_features,
    )

    with open(TRANSCRIPTS_JSON, encoding="utf-8") as f:
        transcripts = json.load(f)

    label_map = {}
    if HAS_GT:
        fn_col = next((c for c in gt_df.columns if c.lower() in ("filename", "file", "name")), None)
        if fn_col and label_col:
            label_map = dict(zip(gt_df[fn_col], gt_df[label_col]))

    rows = []
    for filepath, t in tqdm(transcripts.items(), desc="Extracting text features"):
        text = t.get("text", "")
        words = t.get("words", [])
        filename = t.get("filename", Path(filepath).name)

        text_feats = compute_text_features(text)
        pause_feats = compute_pause_features(words) if words else _empty_pause_features()
        prosodic_feats = compute_prosodic_features(filepath) if os.path.exists(filepath) else _empty_prosodic_features()

        row = {
            "filepath": filepath,
            "filename": filename,
            "label_int": label_map.get(filename, -1),
            "duration_sec": t.get("duration_sec", 0),
            "text": text[:200],
        }
        row.update(text_feats)
        row.update(pause_feats)
        row.update(prosodic_feats)
        rows.append(row)

    feat_df = pd.DataFrame(rows)
    feat_df.to_csv(FEATURES_CSV, index=False)
    print(f"Saved: {FEATURES_CSV} ({len(feat_df)} rows)")
else:
    print(f"Text features already exist: {FEATURES_CSV}")

## 4. WavLM Embeddings (if needed)
Skipped if WavLM CSV already exists or if WavLM weight is 0.

In [ ]:
need_wavlm = not WAVLM_FEAT_CSV.exists()

if need_wavlm:
    import torch
    import librosa
    from transformers import AutoFeatureExtractor, WavLMModel

    print("Loading WavLM-base-plus...")
    fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
    wavlm_model = WavLMModel.from_pretrained("microsoft/wavlm-base-plus")
    wavlm_model = wavlm_model.eval().to("cpu")
    EMBED_DIM = wavlm_model.config.hidden_size

    SR = 16000
    MAX_DUR = 60

    embeddings = []
    fnames = []
    for fp in tqdm(audio_files, desc="Extracting WavLM embeddings"):
        try:
            audio, _ = librosa.load(str(fp), sr=SR, mono=True, duration=MAX_DUR)
            if len(audio) < SR:
                embeddings.append(np.zeros(EMBED_DIM))
            else:
                with torch.no_grad():
                    inputs = fe(audio, sampling_rate=SR, return_tensors="pt", padding=True)
                    outputs = wavlm_model(**inputs)
                    emb = outputs.last_hidden_state[0].mean(dim=0).numpy()
                embeddings.append(emb)
        except Exception as e:
            print(f"  Failed {fp.name}: {e}")
            embeddings.append(np.zeros(EMBED_DIM))
        fnames.append(fp.name)

    embed_cols = [f"wavlm_{i}" for i in range(EMBED_DIM)]
    wl_df = pd.DataFrame(embeddings, columns=embed_cols)
    wl_df["filename"] = fnames
    wl_df["filepath"] = [str(f) for f in audio_files]
    wl_df.to_csv(WAVLM_FEAT_CSV, index=False)
    print(f"Saved: {WAVLM_FEAT_CSV} ({len(wl_df)} rows x {EMBED_DIM} features)")

    del wavlm_model, fe
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    print(f"WavLM features already exist: {WAVLM_FEAT_CSV}")

## 4b. SBERT Embeddings (if needed)
Extracts 384-dim sentence embeddings from transcripts using `all-MiniLM-L6-v2`.

In [ ]:
need_sbert = not SBERT_FEAT_CSV.exists()

if need_sbert:
    if not TRANSCRIPTS_JSON.exists():
        print("WARNING: No transcripts — cannot extract SBERT embeddings.")
    else:
        from sentence_transformers import SentenceTransformer
        sbert_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        SBERT_DIM = 384

        with open(TRANSCRIPTS_JSON, encoding="utf-8") as f:
            transcripts = json.load(f)

        rows = []
        for fp in tqdm(audio_files, desc="SBERT embeddings"):
            fn = fp.name
            text = transcripts.get(fn, {}).get("text", "")

            if text and len(text.strip()) >= 10:
                emb = sbert_model.encode(text, normalize_embeddings=True)
                row = {"filename": fn}
                row.update({f"sbert_{i}": float(emb[i]) for i in range(SBERT_DIM)})
            else:
                row = {"filename": fn}
                row.update({f"sbert_{i}": 0.0 for i in range(SBERT_DIM)})
            rows.append(row)

        sbert_df = pd.DataFrame(rows)
        sbert_df.to_csv(SBERT_FEAT_CSV, index=False)
        print(f"Saved: {SBERT_FEAT_CSV} ({len(sbert_df)} rows, {SBERT_DIM} dims)")
        del sbert_model
else:
    print(f"SBERT features already exist: {SBERT_FEAT_CSV}")

## 6. Stacked Ensemble Inference
Loads WavLM XGBoost + Hybrid XGBoost (SBERT + pause + filler + prosodic), feeds their probabilities through a logistic regression meta-learner.

In [ ]:
# Build master dataframe aligned by filename
filenames = [f.name for f in audio_files]
filepaths = [str(f) for f in audio_files]
master = pd.DataFrame({"filename": filenames, "filepath": filepaths})

# -- Feature column definitions --
PAUSE_FEATURES = [
    "pause_mean", "pause_std", "pause_median", "pause_skew",
    "long_pause_rate", "pause_ratio", "n_pauses", "pause_regularity",
    "pause_before_content_ratio", "pause_before_function_ratio",
    "mid_phrase_pause_rate", "words_per_sec", "articulation_rate",
]
PROSODIC_FEATURES = [
    "f0_mean", "f0_std", "f0_range", "f0_skew", "f0_slope",
    "energy_mean", "energy_std", "speaking_rate_std",
]
HYBRID_HANDCRAFTED = ["filler_rate", "filler_count"] + PAUSE_FEATURES + PROSODIC_FEATURES

# -- WavLM XGBoost scores --
wavlm_proba = np.zeros(len(master))
if WAVLM_FEAT_CSV.exists() and os.path.exists(WAVLM_XGBOOST_MODEL) and os.path.exists(WAVLM_SCALER):
    wl_df = pd.read_csv(WAVLM_FEAT_CSV)
    wl_df = wl_df.set_index("filename").reindex(filenames).reset_index()
    wavlm_cols = [c for c in wl_df.columns if c.startswith("wavlm_")]

    w_model = xgb.XGBClassifier()
    w_model.load_model(WAVLM_XGBOOST_MODEL)
    w_scaler = joblib.load(WAVLM_SCALER)

    X_wl = wl_df[wavlm_cols].fillna(0).values
    X_wl_s = w_scaler.transform(X_wl)
    wavlm_proba = w_model.predict_proba(X_wl_s)[:, 1]
    print(f"WavLM XGBoost: scored {len(wavlm_proba)} files (mean={wavlm_proba.mean():.4f})")
else:
    print("WARNING: WavLM model/features not found.")

# -- Hybrid XGBoost scores (SBERT + pause + filler + prosodic) --
hybrid_proba = np.zeros(len(master))
can_hybrid = (
    SBERT_FEAT_CSV.exists() and FEATURES_CSV.exists()
    and os.path.exists(HYBRID_XGBOOST_MODEL) and os.path.exists(HYBRID_SCALER)
)
if can_hybrid:
    # Load SBERT embeddings
    sbert_df = pd.read_csv(SBERT_FEAT_CSV)
    sbert_df = sbert_df.set_index("filename").reindex(filenames).reset_index()
    sbert_cols = [c for c in sbert_df.columns if c.startswith("sbert_")]

    # Load handcrafted features (filler + pause + prosodic)
    feat_df = pd.read_csv(FEATURES_CSV)
    feat_df = feat_df.set_index("filename").reindex(filenames).reset_index()
    hc_cols = [c for c in HYBRID_HANDCRAFTED if c in feat_df.columns]

    # Build hybrid feature matrix
    X_sbert = sbert_df[sbert_cols].fillna(0).values
    X_hc = feat_df[hc_cols].fillna(0).values
    X_hybrid = np.hstack([X_sbert, X_hc])

    h_model = xgb.XGBClassifier()
    h_model.load_model(HYBRID_XGBOOST_MODEL)
    h_scaler = joblib.load(HYBRID_SCALER)

    X_hybrid_s = h_scaler.transform(X_hybrid)
    hybrid_proba = h_model.predict_proba(X_hybrid_s)[:, 1]
    print(f"Hybrid XGBoost: scored {len(hybrid_proba)} files (mean={hybrid_proba.mean():.4f})")
else:
    print("WARNING: Hybrid model/features not found.")

# -- Stacked ensemble via meta-learner --
if os.path.exists(META_MODEL):
    meta = joblib.load(META_MODEL)
    stack_X = np.column_stack([wavlm_proba, hybrid_proba])
    combined = meta.predict_proba(stack_X)[:, 1]
    print(f"\nMeta-learner loaded: WavLM coef={meta.coef_[0][0]:.3f}, Hybrid coef={meta.coef_[0][1]:.3f}")
else:
    print("WARNING: Meta-model not found, falling back to mean of WavLM + Hybrid")
    combined = 0.5 * wavlm_proba + 0.5 * hybrid_proba

# -- Baseline text model (for comparison) --
text_proba = np.zeros(len(master))
if TEXT_XGBOOST_MODEL and os.path.exists(TEXT_XGBOOST_MODEL) and os.path.exists(TEXT_SCALER):
    TEXT_FEATURES = [
        "filler_rate", "filler_count", "repetition_rate", "repair_rate",
        "ttr", "mattr", "complex_word_rate", "avg_word_length",
        "n_words", "n_unique_words",
        "avg_sentence_length", "std_sentence_length", "fragment_rate", "n_sentences",
        "self_ref_rate", "discourse_marker_rate", "hedge_rate",
        "noun_rate", "verb_rate", "adj_rate",
    ]
    ALL_TEXT = TEXT_FEATURES + PAUSE_FEATURES + PROSODIC_FEATURES
    if FEATURES_CSV.exists():
        if 'feat_df' not in dir():
            feat_df = pd.read_csv(FEATURES_CSV)
            feat_df = feat_df.set_index("filename").reindex(filenames).reset_index()
        text_cols = [c for c in ALL_TEXT if c in feat_df.columns]
        t_model = xgb.XGBClassifier()
        t_model.load_model(TEXT_XGBOOST_MODEL)
        t_scaler = joblib.load(TEXT_SCALER)
        if t_scaler.n_features_in_ == len(text_cols):
            X_text = feat_df[text_cols].fillna(0).values
            X_text_s = t_scaler.transform(X_text)
            text_proba = t_model.predict_proba(X_text_s)[:, 1]
            print(f"Text XGBoost (baseline): scored {len(text_proba)} files (mean={text_proba.mean():.4f})")

master["wavlm_score"] = np.round(wavlm_proba, 4)
master["hybrid_score"] = np.round(hybrid_proba, 4)
master["text_score"] = np.round(text_proba, 4)
master["combined_score"] = np.round(combined, 4)
master["pred_label"] = (combined >= THRESHOLD).astype(int)
master["pred_label_str"] = master["pred_label"].map({1: "cheating", 0: "not cheating"})

print(f"\nArchitecture: Stacked (WavLM + Hybrid -> LogReg)")
print(f"Threshold: {THRESHOLD}")
print(f"Predictions: {(master['pred_label']==1).sum()} cheating, {(master['pred_label']==0).sum()} not cheating")

## 7. Attach GT Labels (if available)

In [ ]:
LABEL_MAP = {
    "read": 1, "Read": 1, "READ": 1, "cheating": 1, "Cheating": 1,
    "reading": 1, "Reading": 1, "yes": 1, "Yes": 1, "Y": 1, "1": 1, 1: 1,
    "spontaneous": 0, "Spontaneous": 0, "not cheating": 0, "Not Cheating": 0,
    "Not cheating": 0, "no": 0, "No": 0, "N": 0, "0": 0, 0: 0,
    "genuine": 0, "Genuine": 0,
}

if HAS_GT:
    fn_col = next((c for c in gt_df.columns if c.lower() in ("filename", "file", "name")), None)
    if fn_col:
        gt_map = {}
        for _, row in gt_df.iterrows():
            raw = row[label_col]
            mapped = LABEL_MAP.get(raw, LABEL_MAP.get(str(raw), -1))
            gt_map[row[fn_col]] = mapped

        master["gt_label"] = master["filename"].map(gt_map).fillna(-1).astype(int)
        labeled = master[master["gt_label"] >= 0]
        print(f"Matched GT labels: {len(labeled)}/{len(master)} files")
        print(f"  Cheating (GT):     {(labeled['gt_label']==1).sum()}")
        print(f"  Not cheating (GT): {(labeled['gt_label']==0).sum()}")
    else:
        print("Could not match GT â€” no filename column found.")
        HAS_GT = False
else:
    master["gt_label"] = -1
    print("No GT labels â€” predictions only.")

## 8. Metrics (if GT available)

In [ ]:
if HAS_GT:
    labeled = master[master["gt_label"] >= 0].copy()
    y_true = labeled["gt_label"].values
    y_pred = labeled["pred_label"].values
    y_score = labeled["combined_score"].values

    print(f"{'='*60}")
    print(f"METRICS (n={len(labeled)}, threshold={THRESHOLD})")
    print(f"Weights: text={W_TEXT}, wavlm={W_WAVLM}, w2v={W_WAV2VEC2}")
    print(f"{'='*60}")
    print(f"\nAccuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"F1:        {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print()

    n_classes_pred = len(set(y_pred))
    n_classes_true = len(set(y_true))
    if n_classes_pred >= 2 and n_classes_true >= 2:
        print(classification_report(y_true, y_pred, target_names=["not cheating", "cheating"]))
    else:
        print(f"WARNING: Only {n_classes_pred} predicted class(es) â€” all predictions are {'cheating' if y_pred[0]==1 else 'not cheating'}.")
        print(f"  Try adjusting the threshold (current: {THRESHOLD}).")

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(f"Confusion Matrix:")
    print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
    print(f"  FN={cm[1,0]}  TP={cm[1,1]}")

    # Misclassifications
    wrong = labeled[labeled["gt_label"] != labeled["pred_label"]]
    fp_df = wrong[wrong["pred_label"] == 1]
    fn_df = wrong[wrong["pred_label"] == 0]
    print(f"\nMisclassifications: {len(wrong)} ({len(fp_df)} FP, {len(fn_df)} FN)")
    if len(wrong) > 0:
        show = ["filename", "gt_label", "pred_label_str", "combined_score",
                "text_score", "wavlm_score", "w2v_score"]
        show = [c for c in show if c in wrong.columns]
        print(wrong[show].to_string(index=False))

    # Threshold sweep
    print(f"\n{'='*60}")
    print("THRESHOLD SWEEP")
    print(f"{'='*60}")
    print(f"{'thresh':>7s} {'prec':>7s} {'recall':>7s} {'f1':>7s} {'flagged':>8s} {'FP':>4s} {'FN':>4s}")
    print("-" * 50)
    best_t_f1, best_t = 0, 0
    for t in np.arange(0.10, 0.91, 0.05):
        preds = (y_score >= t).astype(int)
        f = f1_score(y_true, preds, zero_division=0)
        p = precision_score(y_true, preds, zero_division=0)
        r = recall_score(y_true, preds, zero_division=0)
        n_fp = int(((y_true == 0) & (preds == 1)).sum())
        n_fn = int(((y_true == 1) & (preds == 0)).sum())
        marker = " <-- best" if f > best_t_f1 else ""
        if f > best_t_f1:
            best_t_f1 = f
            best_t = t
        print(f"  {t:.2f}   {p:.4f}  {r:.4f}  {f:.4f}  {preds.sum():>6d}  {n_fp:>3d}  {n_fn:>3d}{marker}")
    print(f"\nBest threshold: {best_t:.2f} (F1={best_t_f1:.4f})")

else:
    print("No GT labels â€” skipping metrics.")

## 8b. Precision–Recall vs Text/WavLM Weight
Sweeps text weight from 0→1 (wavlm = 1−text), shows precision & recall at each weight for multiple thresholds.

In [ ]:
if HAS_GT:
    import matplotlib.pyplot as plt

    labeled = master[master["gt_label"] >= 0].copy()
    y_true = labeled["gt_label"].values
    t_scores = labeled["text_score"].values
    wl_scores = labeled["wavlm_score"].values

    weights_text = np.arange(0.0, 1.01, 0.05)
    thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # ── Left: Precision & Recall vs text weight for each threshold ──
    colors = plt.cm.viridis(np.linspace(0, 1, len(thresholds)))
    for idx, thr in enumerate(thresholds):
        precs, recs = [], []
        for wt in weights_text:
            ww = round(1.0 - wt, 2)
            combined = wt * t_scores + ww * wl_scores
            preds = (combined >= thr).astype(int)
            precs.append(precision_score(y_true, preds, zero_division=0))
            recs.append(recall_score(y_true, preds, zero_division=0))
        axes[0].plot(weights_text, precs, "-", color=colors[idx], label=f"thr={thr:.2f}")
        axes[0].plot(weights_text, recs, "--", color=colors[idx])

    axes[0].set_xlabel("Text Weight (WavLM = 1 - Text)")
    axes[0].set_ylabel("Score")
    axes[0].set_title("Precision (solid) & Recall (dashed) vs Text Weight")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_xlim(0, 1)
    axes[0].set_ylim(0, 1.05)

    # ── Right: Precision vs Recall parametric curve ──
    for idx, thr in enumerate(thresholds):
        precs, recs = [], []
        for wt in weights_text:
            ww = round(1.0 - wt, 2)
            combined = wt * t_scores + ww * wl_scores
            preds = (combined >= thr).astype(int)
            precs.append(precision_score(y_true, preds, zero_division=0))
            recs.append(recall_score(y_true, preds, zero_division=0))
        axes[1].plot(recs, precs, "o-", color=colors[idx], label=f"thr={thr:.2f}", markersize=3)

    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    axes[1].set_title("Precision vs Recall (each point = different text/wavlm weight)")
    axes[1].legend(fontsize=8)
    axes[1].grid(True, alpha=0.3)
    axes[1].set_xlim(0, 1.05)
    axes[1].set_ylim(0, 1.05)

    plt.tight_layout()
    plt.savefig(NB_DIR / f"{folder_name}_pr_weights.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {folder_name}_pr_weights.png")

    # ── Table: best configs sorted by precision (recall >= 0.50) ──
    print(f"\n{'='*70}")
    print("TOP CONFIGS (precision-first, requiring recall >= 0.50)")
    print(f"{'='*70}")
    rows = []
    for wt in np.arange(0.0, 1.01, 0.05):
        ww = round(1.0 - wt, 2)
        for thr in np.arange(0.20, 0.81, 0.05):
            combined = wt * t_scores + ww * wl_scores
            preds = (combined >= thr).astype(int)
            p = precision_score(y_true, preds, zero_division=0)
            r = recall_score(y_true, preds, zero_division=0)
            f = f1_score(y_true, preds, zero_division=0)
            if r >= 0.50:
                rows.append({"w_text": round(wt,2), "w_wavlm": round(ww,2),
                             "threshold": round(thr,2), "precision": round(p,4),
                             "recall": round(r,4), "f1": round(f,4)})
    if rows:
        cfg_df = pd.DataFrame(rows).sort_values("precision", ascending=False)
        print(cfg_df.head(20).to_string(index=False))
    else:
        print("No config achieves recall >= 0.50")
else:
    print("No GT labels - skipping PR curve.")


## 8c. Error Analysis
Deep dive into misclassifications: what features distinguish FP/FN from correct predictions, score distributions, and per-file breakdown to identify patterns.

In [ ]:
if HAS_GT:
    import matplotlib.pyplot as plt

    labeled = master[master["gt_label"] >= 0].copy()
    y_true = labeled["gt_label"].values
    y_pred = labeled["pred_label"].values

    # Classify each sample
    labeled["outcome"] = "TN"
    labeled.loc[(y_true == 1) & (y_pred == 1), "outcome"] = "TP"
    labeled.loc[(y_true == 0) & (y_pred == 1), "outcome"] = "FP"
    labeled.loc[(y_true == 1) & (y_pred == 0), "outcome"] = "FN"

    fp_df = labeled[labeled["outcome"] == "FP"]
    fn_df = labeled[labeled["outcome"] == "FN"]
    tp_df = labeled[labeled["outcome"] == "TP"]
    tn_df = labeled[labeled["outcome"] == "TN"]

    print(f"{'='*70}")
    print(f"ERROR ANALYSIS  (n={len(labeled)}, threshold={THRESHOLD})")
    print(f"TP={len(tp_df)}  TN={len(tn_df)}  FP={len(fp_df)}  FN={len(fn_df)}")
    print(f"{'='*70}")

    # ── 1. Score distributions by outcome ──
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, (score_col, title) in zip(axes, [
        ("text_score", "Text Model Score"),
        ("wavlm_score", "WavLM Model Score"),
        ("combined_score", "Combined Score")
    ]):
        for outcome, color, marker in [("TP", "green", "o"), ("TN", "blue", "o"),
                                        ("FP", "red", "x"), ("FN", "orange", "x")]:
            subset = labeled[labeled["outcome"] == outcome]
            if len(subset) > 0:
                ax.scatter(range(len(subset)), subset[score_col].values,
                          c=color, marker=marker, label=f"{outcome} (n={len(subset)})", alpha=0.7, s=40)
        ax.axhline(y=THRESHOLD, color="black", linestyle="--", alpha=0.5, label=f"threshold={THRESHOLD}")
        ax.set_title(title)
        ax.set_ylabel("Score")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(NB_DIR / f"{folder_name}_error_scores.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ── 2. Text vs WavLM scatter: where do errors live? ──
    fig, ax = plt.subplots(figsize=(10, 8))
    for outcome, color, marker, size in [("TN", "blue", "o", 30), ("TP", "green", "o", 30),
                                          ("FP", "red", "X", 80), ("FN", "orange", "X", 80)]:
        subset = labeled[labeled["outcome"] == outcome]
        if len(subset) > 0:
            ax.scatter(subset["text_score"], subset["wavlm_score"],
                      c=color, marker=marker, s=size, label=f"{outcome} (n={len(subset)})", alpha=0.7)

    # Draw decision boundary line
    if W_TEXT > 0 and W_WAVLM > 0:
        x_line = np.linspace(0, 1, 100)
        y_line = (THRESHOLD - W_TEXT * x_line) / W_WAVLM
        valid = (y_line >= 0) & (y_line <= 1)
        ax.plot(x_line[valid], y_line[valid], "k--", alpha=0.5,
                label=f"boundary ({W_TEXT:.1f}*text+{W_WAVLM:.1f}*wavlm={THRESHOLD})")

    ax.set_xlabel("Text Score")
    ax.set_ylabel("WavLM Score")
    ax.set_title("Text vs WavLM Score - Error Map")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    plt.savefig(NB_DIR / f"{folder_name}_error_map.png", dpi=150, bbox_inches="tight")
    plt.show()

    # ── 3. Feature comparison: FN vs TP ──
    print(f"\n{'='*70}")
    print("WHY ARE WE MISSING CHEATERS? (FN vs TP feature comparison)")
    print(f"{'='*70}")

    if FEATURES_CSV.exists() and len(fn_df) > 0 and len(tp_df) > 0:
        feat_df = pd.read_csv(FEATURES_CSV)
        feat_df = feat_df.set_index("filename")

        key_feats = ["filler_rate", "repetition_rate", "ttr", "mattr",
                     "avg_word_length", "n_words", "pause_ratio",
                     "long_pause_rate", "words_per_sec", "pause_mean",
                     "f0_std", "energy_std", "speaking_rate_std",
                     "hedge_rate", "discourse_marker_rate", "self_ref_rate"]
        key_feats = [f for f in key_feats if f in feat_df.columns]

        tp_feats = feat_df.loc[feat_df.index.isin(tp_df["filename"])][key_feats]
        fn_feats = feat_df.loc[feat_df.index.isin(fn_df["filename"])][key_feats]
        tn_feats = feat_df.loc[feat_df.index.isin(tn_df["filename"])][key_feats]

        print(f"\nFN (missed cheaters, n={len(fn_feats)}):  look like genuine to the model")
        print(f"TP (caught cheaters, n={len(tp_feats)}):    model correctly flagged")
        print(f"TN (genuine speakers, n={len(tn_feats)}):   model correctly passed\n")

        comp = pd.DataFrame({
            "TP_mean": tp_feats.mean(),
            "FN_mean": fn_feats.mean(),
            "TN_mean": tn_feats.mean(),
            "FN_vs_TP": (fn_feats.mean() - tp_feats.mean()),
        })
        comp["FN_closer_to"] = ["genuine" if abs(fn - tn) < abs(fn - tp) else "cheating"
                                 for fn, tn, tp in zip(fn_feats.mean(), tn_feats.mean(), tp_feats.mean())]
        comp["FN_vs_TP_pct"] = ((comp["FN_mean"] - comp["TP_mean"]) / (comp["TP_mean"].abs() + 1e-8) * 100).round(1)

        print("Feature comparison (large % = why model misses them):")
        print(comp.sort_values("FN_vs_TP_pct", key=abs, ascending=False).to_string())

        print(f"\n--- KEY INSIGHTS ---")
        genuine_like = comp[comp["FN_closer_to"] == "genuine"].index.tolist()
        if genuine_like:
            print(f"FN look like GENUINE speakers on: {', '.join(genuine_like)}")
        cheating_like = comp[comp["FN_closer_to"] == "cheating"].index.tolist()
        if cheating_like:
            print(f"FN look like CHEATING on: {', '.join(cheating_like)} (model still missed!)")
    else:
        if len(fn_df) == 0:
            print("No false negatives - recall is 100%!")
        else:
            print("Cannot compare features - CSV not found or no TP samples.")

    # ── 4. Per-file FN breakdown ──
    if len(fn_df) > 0:
        print(f"\n{'='*70}")
        print(f"FALSE NEGATIVES - Individual Files (missed cheaters)")
        print(f"{'='*70}")
        show = ["filename", "text_score", "wavlm_score", "combined_score"]
        show = [c for c in show if c in fn_df.columns]
        print(fn_df[show].sort_values("combined_score", ascending=False).to_string(index=False))

        fn_text_high = (fn_df["text_score"] >= 0.5).sum()
        fn_wavlm_high = (fn_df["wavlm_score"] >= 0.5).sum()
        fn_neither = ((fn_df["text_score"] < 0.5) & (fn_df["wavlm_score"] < 0.5)).sum()
        print(f"\nOf {len(fn_df)} FN files:")
        print(f"  Text says cheating (score>=0.5):  {fn_text_high} ({fn_text_high/len(fn_df)*100:.0f}%)")
        print(f"  WavLM says cheating (score>=0.5): {fn_wavlm_high} ({fn_wavlm_high/len(fn_df)*100:.0f}%)")
        print(f"  Neither says cheating:            {fn_neither}")

    # ── 5. Per-file FP breakdown ──
    if len(fp_df) > 0:
        print(f"\n{'='*70}")
        print(f"FALSE POSITIVES - Individual Files (wrongly flagged)")
        print(f"{'='*70}")
        show = ["filename", "text_score", "wavlm_score", "combined_score"]
        show = [c for c in show if c in fp_df.columns]
        print(fp_df[show].sort_values("combined_score", ascending=False).to_string(index=False))

        fp_text_high = (fp_df["text_score"] >= 0.5).sum()
        fp_wavlm_high = (fp_df["wavlm_score"] >= 0.5).sum()
        print(f"\nOf {len(fp_df)} FP files:")
        print(f"  Text triggered (score>=0.5):  {fp_text_high} ({fp_text_high/len(fp_df)*100:.0f}%)")
        print(f"  WavLM triggered (score>=0.5): {fp_wavlm_high} ({fp_wavlm_high/len(fp_df)*100:.0f}%)")

else:
    print("No GT labels - skipping error analysis.")


## 9. All Predictions

In [ ]:
# Display
show_cols = ["filename"]
if HAS_GT:
    show_cols.append("gt_label")
show_cols += ["pred_label_str", "combined_score", "text_score", "wavlm_score", "w2v_score"]
if "fused_pred_str" in master.columns:
    show_cols += ["fused_pred_str", "fused_score"]
show_cols = [c for c in show_cols if c in master.columns]

print(f"\nAll predictions ({len(master)} files):")
print(master[show_cols].to_string(index=False))

# Save â€” one CSV with both methods clearly labeled
out_path = NB_DIR / f"{folder_name}_predictions.csv"
save_cols = ["filename", "filepath"]
if HAS_GT:
    save_cols.append("gt_label")
# Weighted columns prefixed
save_cols += ["pred_label_str", "combined_score", "text_score", "wavlm_score", "w2v_score"]
# Fused columns (if available)
if "fused_pred_str" in master.columns:
    save_cols += ["fused_pred_str", "fused_score"]
save_cols = [c for c in save_cols if c in master.columns]

# Rename for clarity in CSV
out_df = master[save_cols].copy()
out_df = out_df.rename(columns={
    "pred_label_str": "weighted_prediction",
    "combined_score": "weighted_score",
    "fused_pred_str": "fused_prediction",
})

out_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

## 10. Quick Compare: Different Weight Configs (if GT available)
Runs multiple weight configurations and shows which is best.

In [ ]:
if HAS_GT:
    labeled = master[master["gt_label"] >= 0]
    y_true = labeled["gt_label"].values
    t_scores = labeled["text_score"].values
    wl_scores = labeled["wavlm_score"].values
    w2_scores = labeled["w2v_score"].values

    configs = []
    # Grid search over weight triplets
    for wt in np.arange(0, 1.01, 0.1):
        for ww in np.arange(0, 1.01 - wt, 0.1):
            w2 = round(1.0 - wt - ww, 1)
            if w2 < 0:
                continue
            combined = round(wt, 1) * t_scores + round(ww, 1) * wl_scores + round(w2, 1) * w2_scores
            # Try multiple thresholds
            for thr in [0.35, 0.40, 0.45, 0.50, 0.55]:
                preds = (combined >= thr).astype(int)
                f = f1_score(y_true, preds, zero_division=0)
                p = precision_score(y_true, preds, zero_division=0)
                r = recall_score(y_true, preds, zero_division=0)
                configs.append({
                    "w_text": round(wt, 1), "w_wavlm": round(ww, 1),
                    "w_w2v": round(w2, 1), "threshold": thr,
                    "f1": round(f, 4), "prec": round(p, 4), "recall": round(r, 4),
                })

    configs_df = pd.DataFrame(configs).sort_values("f1", ascending=False)
    print("Top 15 weight+threshold configurations:")
    print(configs_df.head(15).to_string(index=False))
else:
    print("No GT labels â€” cannot compare configurations.")

## 11. Fused Model Inference (Text + WavLM â†’ single XGBoost)

Instead of weighted voting, concatenates text features (41) + WavLM embeddings (768) = 809 features into a single XGBoost.
Requires both text features and WavLM features to exist. Uses `FUSED_XGBOOST_MODEL` and `FUSED_SCALER` from config.

In [ ]:
can_fuse = (
    FEATURES_CSV.exists() and WAVLM_FEAT_CSV.exists()
    and FUSED_XGBOOST_MODEL and os.path.exists(FUSED_XGBOOST_MODEL)
    and FUSED_SCALER and os.path.exists(FUSED_SCALER)
)

if can_fuse:
    print("Running fused model (text + WavLM concatenated)...")

    # Load text features
    TEXT_FEATURES = [
        "filler_rate", "filler_count", "repetition_rate", "repair_rate",
        "ttr", "mattr", "complex_word_rate", "avg_word_length",
        "n_words", "n_unique_words",
        "avg_sentence_length", "std_sentence_length", "fragment_rate", "n_sentences",
        "self_ref_rate", "discourse_marker_rate", "hedge_rate",
        "noun_rate", "verb_rate", "adj_rate",
    ]
    PAUSE_FEATURES = [
        "pause_mean", "pause_std", "pause_median", "pause_skew",
        "long_pause_rate", "pause_ratio", "n_pauses", "pause_regularity",
        "pause_before_content_ratio", "pause_before_function_ratio",
        "mid_phrase_pause_rate", "words_per_sec", "articulation_rate",
    ]
    PROSODIC_FEATURES = [
        "f0_mean", "f0_std", "f0_range", "f0_skew", "f0_slope",
        "energy_mean", "energy_std", "speaking_rate_std",
    ]
    ALL_TEXT = TEXT_FEATURES + PAUSE_FEATURES + PROSODIC_FEATURES

    feat_df = pd.read_csv(FEATURES_CSV)
    feat_df = feat_df.set_index("filename").reindex(filenames).reset_index()
    text_cols = [c for c in ALL_TEXT if c in feat_df.columns]

    wl_df = pd.read_csv(WAVLM_FEAT_CSV)
    wl_df = wl_df.set_index("filename").reindex(filenames).reset_index()
    wavlm_cols = [c for c in wl_df.columns if c.startswith("wavlm_")]

    X_text = feat_df[text_cols].fillna(0).values
    X_wavlm = wl_df[wavlm_cols].fillna(0).values
    X_fused = np.hstack([X_text, X_wavlm])

    f_model = xgb.XGBClassifier()
    f_model.load_model(FUSED_XGBOOST_MODEL)
    f_scaler = joblib.load(FUSED_SCALER)

    if f_scaler.n_features_in_ != X_fused.shape[1]:
        print(f"WARNING: Fused scaler expects {f_scaler.n_features_in_} features, got {X_fused.shape[1]}. Skipping.")
    else:
        X_fused_s = f_scaler.transform(X_fused)
        fused_proba = f_model.predict_proba(X_fused_s)[:, 1]
        fused_preds = (fused_proba >= FUSED_THRESHOLD).astype(int)

        master["fused_score"] = np.round(fused_proba, 4)
        master["fused_pred"] = fused_preds
        master["fused_pred_str"] = master["fused_pred"].map({1: "cheating", 0: "not cheating"})

        print(f"Fused model: {len(fused_proba)} files (mean={fused_proba.mean():.4f})")
        print(f"Threshold: {FUSED_THRESHOLD}")
        print(f"Predictions: {fused_preds.sum()} cheating, {(fused_preds==0).sum()} not cheating")

        # Metrics if GT available
        if HAS_GT:
            labeled = master[master["gt_label"] >= 0]
            y_true = labeled["gt_label"].values
            y_fused = labeled["fused_pred"].values
            y_fscore = labeled["fused_score"].values

            print(f"\n{'='*60}")
            print(f"FUSED MODEL METRICS (n={len(labeled)}, threshold={FUSED_THRESHOLD})")
            print(f"{'='*60}")
            print(f"Accuracy:  {accuracy_score(y_true, y_fused):.4f}")
            print(f"F1:        {f1_score(y_true, y_fused, zero_division=0):.4f}")
            print(f"Precision: {precision_score(y_true, y_fused, zero_division=0):.4f}")
            print(f"Recall:    {recall_score(y_true, y_fused, zero_division=0):.4f}")
            print()
            print(classification_report(y_true, y_fused, target_names=["not cheating", "cheating"]))

            cm = confusion_matrix(y_true, y_fused)
            print(f"Confusion Matrix:")
            print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
            print(f"  FN={cm[1,0]}  TP={cm[1,1]}")

            # Threshold sweep for fused
            print(f"\nFused threshold sweep:")
            print(f"{'thresh':>7s} {'prec':>7s} {'recall':>7s} {'f1':>7s} {'FP':>4s} {'FN':>4s}")
            print("-" * 40)
            best_ff1, best_ft = 0, 0
            for t in np.arange(0.10, 0.91, 0.05):
                preds = (y_fscore >= t).astype(int)
                f = f1_score(y_true, preds, zero_division=0)
                p = precision_score(y_true, preds, zero_division=0)
                r = recall_score(y_true, preds, zero_division=0)
                n_fp = int(((y_true == 0) & (preds == 1)).sum())
                n_fn = int(((y_true == 1) & (preds == 0)).sum())
                marker = " <-- best" if f > best_ff1 else ""
                if f > best_ff1:
                    best_ff1 = f
                    best_ft = t
                print(f"  {t:.2f}   {p:.4f}  {r:.4f}  {f:.4f}  {n_fp:>3d}  {n_fn:>3d}{marker}")
            print(f"\nBest fused threshold: {best_ft:.2f} (F1={best_ff1:.4f})")

            # Compare weighted vs fused
            w_f1 = f1_score(y_true, labeled["pred_label"].values, zero_division=0)
            print(f"\n{'='*60}")
            print(f"COMPARISON: Weighted vs Fused")
            print(f"  Weighted ensemble: F1={w_f1:.4f} (threshold={THRESHOLD})")
            print(f"  Fused model:       F1={f1_score(y_true, y_fused, zero_division=0):.4f} (threshold={FUSED_THRESHOLD})")
            print(f"{'='*60}")

else:
    missing = []
    if not FEATURES_CSV.exists(): missing.append("text features CSV")
    if not WAVLM_FEAT_CSV.exists(): missing.append("WavLM features CSV")
    if not FUSED_XGBOOST_MODEL or not os.path.exists(FUSED_XGBOOST_MODEL): missing.append("fused XGBoost model")
    if not FUSED_SCALER or not os.path.exists(FUSED_SCALER): missing.append("fused scaler")
    print(f"Cannot run fused model â€” missing: {', '.join(missing)}")